In [93]:
import ast
import math
from collections import Counter
from igraph import Graph
from itertools import combinations

#############################################################
# LOAD RULES
#############################################################

#############################################################
# JACCARD SIMILARITY
#############################################################

def jaccard_similarity(rule1, rule2):

    s1 = set(rule1)
    s2 = set(rule2)

    inter = len(s1.intersection(s2))
    union = len(s1.union(s2))

    if union == 0:
        return 0

    return inter / union

#############################################################
# COMMUNITY DETECTION
#############################################################

def detect_communities(graph):

    communities = graph.community_multilevel(
        weights=graph.es["weight"]
    )

    graph.vs["community"] = communities.membership

    return communities

#############################################################
# RULE GRAPH
#############################################################

def build_rule_graph(rules, threshold=0.5):

    g = Graph()

    g.add_vertices(len(rules))

    edges = []
    weights = []

    for i, j in combinations(range(len(rules)), 2):

        sim = jaccard_similarity(
            rules[i],
            rules[j]
        )

        if sim >= threshold:

            edges.append((i, j))
            weights.append(sim)

    g.add_edges(edges)

    g.es["weight"] = weights

    return g

def load_rules(file_name):

    rules = []

    with open(file_name, "r") as f:

        for line in f:

            line = line.strip()

            if line == "":
                continue

            rule = ast.literal_eval(line)

            attrs = []

            for att in rule[1]:

                attrs.append(
                    (att[0], str(att[1]))
                )

            rules.append(attrs)

    return rules


#############################################################
# WSC
#############################################################

def compute_wsc(rules):

    return sum(len(rule) for rule in rules)


#############################################################
# GLOBAL FREQUENCIES
#############################################################

def compute_global_frequencies(rules):

    counter = Counter()

    total_pairs = 0

    for rule in rules:

        for pair in rule:

            counter[pair] += 1
            total_pairs += 1

    frequencies = {}

    for pair, count in counter.items():

        frequencies[pair] = count / total_pairs

    return frequencies

#############################################################
# COMMUNITY REUSE FACTOR
#############################################################

def compute_reuse_factor(rules, membership):

    reuse = {}

    n_communities = max(membership) + 1

    for cid in range(n_communities):

        unique_pairs = set()

        total_pairs = 0

        for idx, rule in enumerate(rules):

            if membership[idx] != cid:
                continue

            total_pairs += len(rule)

            for pair in rule:
                unique_pairs.add(pair)

        if total_pairs == 0:

            reuse[cid] = 1.0

        else:

            reuse[cid] = len(unique_pairs) / total_pairs

    return reuse

#############################################################
# SHANNON RARITY
#############################################################

def rarity_rule(rule, frequencies):

    rarity = 0

    for pair in rule:

        rarity += -math.log2(
            frequencies[pair]
        )

    return rarity / len(rule)


#############################################################
# NEW METRIC
#############################################################

def compute_new_metric(
        rules,
        membership,
        alpha=1,
        beta=0.05,
        gamma=1.0
):

    frequencies = compute_global_frequencies(
        rules
    )

    reuse_factor = compute_reuse_factor(
    rules,
    membership
)

    complexity = 0

    details = []

    for idx, rule in enumerate(rules):

        size = len(rule)

        rarity = rarity_rule(
            rule,
            frequencies
        )

        community = membership[idx]

        reuse = reuse_factor[community]

        score = (
            (size ** alpha)
            *
            (1 + beta * rarity)
            *
            (1 + gamma * (reuse - 1))
        )

        complexity += score

        details.append({

            "rule": idx,

            "size": size,

            "rarity": rarity,

            "complexity": score

        })

    return complexity, details


#############################################################
# PRINT COMPARISON
#############################################################

def compare_metrics(file_name):

    rules = load_rules(file_name)

    ####################################################
    # RULE GRAPH
    ####################################################

    graph = build_rule_graph(
        rules,
        threshold=0.5
    )

    ####################################################
    # COMMUNITY DETECTION
    ####################################################

    communities = detect_communities(graph)
    print("\nRule Graph")
    print("---------------------------")
    print("Nodes:", graph.vcount())
    print("Edges:", graph.ecount())
    print("Density:", round(graph.density(),4))

    print("\nCommunities:", len(communities))

    sizes = [len(c) for c in communities]

    print("Community sizes:", sizes)

    wsc = compute_wsc(rules)

    new_metric, details = compute_new_metric(
        rules,
        communities.membership,
        alpha=1.5,
        beta=0.5,
        gamma=0.5
    )

    print("=" * 60)

    print("Policy")

    print("=" * 60)

    print("Rules:", len(rules))

    print("WSC:", round(wsc, 2))

    print("New Metric:", round(new_metric, 2))

    print()

    print("=" * 60)

    print("Per-rule analysis")

    print("=" * 60)



#############################################################

if __name__ == "__main__":

    compare_metrics("/Users/ddiaz/Documents/code/phd-thesis-lab/17_NuevoAnalisisP1/reglas_abac.txt")


Rule Graph
---------------------------
Nodes: 75
Edges: 27
Density: 0.0097

Communities: 56
Community sizes: [3, 2, 6, 4, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 2, 3, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Policy
Rules: 75
WSC: 320
New Metric: 3205.6

Per-rule analysis
